# Prepare Atomic parameters

In [ ]:
import numpy as np 
from ase.dft.kpoints import * 
from ase.io import read,write
from ase import Atoms
from ase.build import bulk
from ase.visualize.plot import plot_atoms
from ase.build import mx2
import matplotlib.pyplot as plt
import spglib
from ase.spacegroup import Spacegroup
from ase.spacegroup.symmetrize import check_symmetry

def atom_pos(structure):
    struct_file = structure
    struct_pos = struct_file.get_positions()
    struct_lbl =  structure.get_chemical_symbols()

    kpstr = '\n'.join(['\t'.join([symbol] + 
                    ['{:.10f}'.format(coord) for coord in row]) for symbol, row in zip(struct_lbl, struct_pos)])
    print("\nATOMIC_POSITIONS (angstrom)")
    print(kpstr)
    print(f"No of atoms:{len(struct_pos)}")


def cell_geo(structure):
    print("CELL_PARAMETERS (angstrom)")
    for i in structure.cell:
        print(f"   {i[0]:.9f}   {i[1]:.9f}   {i[2]:.9f}")

#sb2te3_monolayer = read("1Sb2Te3-1-2.json")
# Parámetros de la celda
cell = [
    [4.310483556994066,0.0, 0.0],
    [-2.1552417784970327, 3.732952154669093, 0.0],
    [0.0, 0.0, 22.162984280000003],  # Incluye un vacío de 20 Å en la dirección z
]

# Posiciones atómicas y símbolos
positions = [
    [ 2.48043809,1.81924878,13.07535672 ],  # Sb
    [ 2.47747224,1.82045054,7.39027067 ],  # Te
    [0.32454061,0.57172306,9.08742515 ],  # Sb
    [0.32380581,0.57699184,14.77255646 ],  # Te
    [0.32269052,3.06650183,11.08142752 ],  # Te
]

symbols = ["Sb", "Te", "Sb", "Te", "Te"]

# Construir la estructura usando ASE
sb2te3_monolayer = Atoms(
    symbols=symbols,
    positions=positions,
    cell=cell,
    pbc=[True, True, False]  # Periódico en x e y; no periódico en z
)
# sb2te3_monolayer*=(5,5,1)
initial_symmetry  = check_symmetry(sb2te3_monolayer,verbose=False)['hall']
fig, ax = plt.subplots(figsize=(5,5))
plot_atoms(sb2te3_monolayer,ax, radii=0.35, rotation='0x,0y,0z',)
plt.title(f"Initial symmetry:{initial_symmetry}")
plt.axis('off')
plt.show()
cell_geo(sb2te3_monolayer)
Sb2Te3_cell = sb2te3_monolayer.get_cell()
atom_pos(sb2te3_monolayer)
sb2te3_monolayer.cell.bandpath()
# write('sb2te3_unit_2.cif', Sb2Te3)


## Add Strain

In [ ]:
from atomistiico.aiico_qe import Strain
apply_strain = Strain(sb2te3_monolayer)
strains = 1*np.arange(0,30,1) # De -5% a 5% en 11 pasos
structures=[]
les = ['a','b','c']
for i, strain in enumerate(strains):
    struct_strain= apply_strain.uniaxial(strain,0)
    initial_symmetry  = check_symmetry(struct_strain,verbose=False)['international']
    print(f"Deformación: {strain/100:.2%}")
    for i,j in zip(les,struct_strain.cell.lengths()):
        print(f"{i}         = {j}")
    #cell_geo(struct_strain)
    #atom_pos(struct_strain)
    structures.append(struct_strain)
    #print("\n")    
fig, ax = plt.subplots(figsize=(5,5))
for atoms in structures:
    plot_atoms(atoms, ax, radii=0.35, rotation='0x,0y,0z',)
    plt.axis('off')

# Animation 

In [ ]:
import numpy as np
import glob
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation
plt.rcParams["animation.html"] = "jshtml"
from ase.io.pov import write_pov
from ase.data.colors import jmol_colors as chemical_colors
from tqdm import tqdm
import os  

import re
atom_species = structures[0].get_chemical_symbols()
cell_struct  = structures[0].get_cell()
atom_pos =  structures[0].get_positions()
bonds_length = 5
bondatoms = []
symbols = atom_species
prefix = "Sb2Te3-uatsx-uc"
new_pos = structures[0]

def make_color_list(atoms, color_dict):
    per_atom_list = []
    for z, symbol in zip(atoms.get_atomic_numbers(), atoms.symbols):
        if symbol in color_dict.keys():
            per_atom_list.append(color_dict[symbol])
        else:
            per_atom_list.append(chemical_colors[z])
    return per_atom_list

def find_bonds(atoms,symbols):
    bondatoms = []
    for i in range(len(atoms)):
        for j in range(i):
            if (symbols[i] != symbols[j] == symbols[0] and new_pos.get_distance(i, j) < bonds_length):
                bondatoms.append((i, j))
            elif (symbols[i] != symbols[j] == symbols[1] and new_pos.get_distance(i, j) < bonds_length):
                bondatoms.append((i, j))
    return bondatoms

color_dict_rgb255 = {
    "Sb": [158,99,181],
    'Te': [240,141,6]
}
color_dict = {}
for symbol in color_dict_rgb255:
    color_dict[symbol] = [val / 255 for val in color_dict_rgb255[symbol]]

for l in range(len(structures)):
    new_pos=structures[l]
    bondatoms = find_bonds(structures[l],symbols)
    color_list = make_color_list(structures[l], color_dict)
    pov_name = f'{prefix}-{l}' + '.pov'
    renderer = write_pov(pov_name, new_pos,
                        rotation=('0x,5y,0z'),
                        radii=0.4,
                        show_unit_cell=0,
                        povray_settings=dict(transparent=True,
                                            camera_type='perspective',
                                            camera_dist=100,
                                            colors=color_list,
                                            canvas_width=1080,
                                            bondlinewidth=0.13,
                                            bondatoms=bondatoms,
                                            ))
    renderer.render()
    print(f"Create {prefix}-{l}")

ini_files = glob.glob("*.ini")
pov_files = glob.glob("*.pov")
for i in tqdm(ini_files, desc="Delete ini files"):
    os.remove(i)
for i in tqdm(pov_files, desc="Delete pov files"):
    os.remove(i)

In [ ]:
import glob
from matplotlib.animation import FuncAnimation, PillowWriter
import matplotlib
files = sorted(glob.glob("Sb2Te3-uatsx-uc*.png"))
def extract_number(filename):
    match = re.search(r'-([0-9]+)\.png$', filename)
    if match:
        return int(match.group(1))
    return -1  # Devuelve -1 para los nombres que no contienen un número

# Ordenar los archivos utilizando la función extract_number
sorted_files = sorted(files, key=extract_number)
print(f"No. of frames:{len(sorted_files)}")
pil_img = Image.open(sorted_files[0])
img0 = np.array(pil_img)
img0_height, img0_width = img0.shape[:2]

dpi = plt.rcParams['figure.dpi']  # Obtener DPI predeterminado de Matplotlib
fig2, ax1 = plt.subplots(figsize=(img0_width / dpi, img0_height / dpi))
ax1.set_xlim(0, img0.shape[1])
ax1.set_ylim(img0.shape[0], 0)
ax1.set_axis_off() 
ax1.imshow(img0)
def anim_render(frame):
    if frame < len(sorted_files):
        idx = frame
    else:
        idx = 2 * len(sorted_files) - frame - 1 # Índice para el efecto de reversa
    # Cargar la imagen
    pil_img = Image.open(sorted_files[idx])
    img = np.array(pil_img)
    ax1.clear()
    ax1.set_xlim(0, img0.shape[1])
    ax1.set_ylim(img0.shape[0], 0)
    ax1.imshow(img)
    #ax1.set_title(fr"Tensile Strain $x$:{' ' * 2}{idx:02d}\%", fontsize=45)
    ax1.set_axis_off()     
    
total_frames = 2 * len(sorted_files)   # Hacia adelante y luego hacia atrás
animation = FuncAnimation(fig2, anim_render, frames=total_frames)
animation   

In [ ]:
animation.save('Sb2Te3-ucx.gif',writer='imagemagick', fps=30)